In [2]:
# ============================================
# 🧰 Install required libraries
# ============================================
!pip install pandas langdetect matplotlib requests tensorflow-datasets --quiet
!pip install apache_beam --quiet

# ============================================
# 🧹 C4-style cleaning via wrappers around c4_utils + watermark detection
# ============================================

import re
import json
import hashlib
import requests
import pandas as pd
import matplotlib.pyplot as plt

# --- NLTK sentence tokenizer (still used for post-dedupe checks) ---
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

# --- Langdetect, deterministic seed (C4 style) ---
from langdetect import detect_langs, DetectorFactory
DetectorFactory.seed = 0

# --- Import C4 utilities from TFDS ---
from tensorflow_datasets.text import c4_utils
PageFeatures = c4_utils.PageFeatures  # convenience alias

# ============================================
# ⚙️ Config (match TFDS C4 clean=True, en)
# ============================================

CONFIG = {
    "target_lang": "en",
    "clean": True,
    "dedupe": True,
    "badwords_filter_fraction": 1.0,
    "min_words_per_line": 5,
    "min_num_sentences": 3,
    "max_word_length": 1000,
    "max_length_chars": int(1.9e5),
    "line_delimiter": "\n",
}

# ============================================
# 1️⃣ Load your data + watermark reference characters
# ============================================

data_path = "/content/final_uniform_replace.jsonl"
mytext_path = "/content/myText.txt"

# --- Load dataset ---
records = [json.loads(l) for l in open(data_path, "r", encoding="utf-8")]
df = pd.DataFrame(records)
df = df[df.get("is_watermarked", False) == True].copy()
texts = df["watermarked"].astype(str).tolist()
print(f"✅ Loaded {len(texts)} watermarked documents.")

# --- Load invisible characters from myText.txt ---
with open(mytext_path, "r", encoding="utf-8") as f:
    chars = f.read()

ZWC = [c for c in chars if not c.isprintable() and c != "\n"]
ZWC_CODES = [f"U+{ord(c):04X}" for c in ZWC]
print(f"💧 Loaded {len(ZWC)} invisible watermark characters:")
print(", ".join(ZWC_CODES[:15]) + (" ..." if len(ZWC_CODES) > 15 else ""))

# ============================================
# 2️⃣ Core cleaning utilities via c4_utils wrappers
# ============================================

def normalize_text(raw_text: str) -> str:
    """Simple normalization to ensure string input."""
    if not isinstance(raw_text, str):
        raw_text = str(raw_text)
    return raw_text.strip()


# --- Wrapper around c4_utils.clean_page, disabling sentence-count filter ---
CITATION_REGEX = re.compile(r"\[\d*\]|\[edit\]|\[citation needed\]")

def c4_clean_page_wrapper(text: str) -> str | None:
    """
    Wraps the official C4 clean_page logic,
    but disables the minimum-number-of-sentences filter
    by setting min_num_sentences=0.
    """
    page = PageFeatures(text=text)
    cleaned_iter = c4_utils.clean_page(
        page,
        citation_regex=CITATION_REGEX,
        min_words_per_line=CONFIG["min_words_per_line"],
        min_num_sentences=0,
        max_word_length=CONFIG["max_word_length"],
        line_delimiter=CONFIG["line_delimiter"],
    )
    cleaned_pages = list(cleaned_iter)
    if not cleaned_pages:
        return None
    return cleaned_pages[0].text


# --- Wrapper around c4_utils.is_valid_length ---
def is_valid_length_wrapper(
    text: str,
    max_length: int = CONFIG["max_length_chars"],
) -> bool:
    page = PageFeatures(text=text)
    return c4_utils.is_valid_length(page, max_length=max_length)


# --- Our own MD5 hash for lines (not from c4_utils, used for local/global dedupe) ---
def md5_hash_line(line: str) -> str:
    return hashlib.md5(line.strip().lower().encode("utf-8")).hexdigest()


def dedupe_lines_c4_style(text: str, line_delimiter: str = CONFIG["line_delimiter"]) -> str:
    """
    Simple intra-document line deduplication (local dedupe).

    Note: This is NOT the full Beam-based remove_duplicate_text from C4,
    but uses the same idea of MD5 hashing per line.
    """
    seen, kept = set(), []
    for line in text.split(line_delimiter):
        h = md5_hash_line(line)
        if h not in seen:
            seen.add(h)
            kept.append(line)
    return line_delimiter.join(kept).strip()


# --- Wrapper around c4_utils.detect_english ---
def detect_english_wrapper(text: str, min_prob: float = 0.99) -> bool:
    """
    Uses the official C4 detect_english logic via c4_utils.

    Returns True iff the page is accepted as English.
    """
    page = PageFeatures(text=text)
    # detect_english is a generator that yields PageFeatures if accepted
    for _ in c4_utils.detect_english(page, min_probability=min_prob):
        return True
    return False


# --- Badwords filter: use c4_utils.get_badwords_filter_fn for English ---
def load_ldnoobw_badwords_en() -> set[str]:
    try:
        url = (
            "https://raw.githubusercontent.com/LDNOOBW/"
            "List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words/master/en"
        )
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        return {w.strip() for w in resp.text.splitlines() if w.strip()}
    except Exception:
        # Fallback small set
        return {"sex", "nsfw", "badword"}


EN_BADWORDS = load_ldnoobw_badwords_en()
_BADWORDS_DICT = {"en": list(EN_BADWORDS)}

_BADWORDS_FILTER_FN = c4_utils.get_badwords_filter_fn(
    badwords=_BADWORDS_DICT,
    filter_fraction=CONFIG["badwords_filter_fraction"],
)

def badwords_filter_en_wrapper(text: str) -> bool:
    """
    Wraps the C4 badwords filter for English using get_badwords_filter_fn.

    Returns True if the page is kept, False if filtered out.
    """
    page = PageFeatures(text=text, language="en")
    return _BADWORDS_FILTER_FN(page)


# ============================================
# 3️⃣ Full C4-like clean pipeline (using wrappers)
# ============================================

def full_c4_clean_single(text: str, doc_id=None, debug_log=None) -> str | None:
    if not text or not isinstance(text, str):
        if debug_log is not None:
            debug_log.append((doc_id, "empty_or_invalid"))
        return None

    # Basic normalization
    text = normalize_text(text)

    # Length filter (wrapper around c4_utils.is_valid_length)
    if not is_valid_length_wrapper(text):
        if debug_log is not None:
            debug_log.append((doc_id, "too_long"))
        return None

    # Core cleaning (wrapper around c4_utils.clean_page)
    text = c4_clean_page_wrapper(text)
    if not text:
        if debug_log is not None:
            debug_log.append((doc_id, "line_filter"))
        return None

    # Intra-document dedupe (local line dedupe, our own implementation)
    if CONFIG["dedupe"]:
        text = dedupe_lines_c4_style(text)
        if not text:
            if debug_log is not None:
                debug_log.append((doc_id, "empty_after_dedupe"))
            return None
        # # Extra sentence count check after dedupe (similar spirit to C4)
        # if len(sent_tokenize(text)) < CONFIG["min_num_sentences"]:
        #     if debug_log is not None:
        #         debug_log.append((doc_id, "too_few_sentences"))
        #     return None

    # Badwords filtering (wrapper around c4_utils.get_badwords_filter_fn)
    if not badwords_filter_en_wrapper(text):
        if debug_log is not None:
            debug_log.append((doc_id, "badwords"))
        return None

    # English detection (wrapper around c4_utils.detect_english)
    if not detect_english_wrapper(text):
        if debug_log is not None:
            debug_log.append((doc_id, "language_not_en"))
        return None

    if debug_log is not None:
        debug_log.append((doc_id, "passed"))

    return text


# ============================================
# 4️⃣ Cleaning execution + simple global dedupe
# ============================================

global_seen_hashes: set[str] = set()
debug_log: list[tuple[int, str]] = []
cleaned_with_id: list[tuple[int, str]] = []


def remove_global_duplicates_keep_id(doc_id: int, text: str):
    """
    Simple global line deduplication across documents.

    NOTE: This is still our own simplified global dedupe,
    NOT the full Beam-based remove_duplicate_text from C4.
    """
    new_lines = []
    for line in text.splitlines():
        h = md5_hash_line(line)
        if h in global_seen_hashes:
            continue
        global_seen_hashes.add(h)
        new_lines.append(line)
    if not new_lines:
        return None
    return (doc_id, "\n".join(new_lines))


for i, t in enumerate(texts):
    cleaned = full_c4_clean_single(t, doc_id=i, debug_log=debug_log)
    if cleaned:
        kept = remove_global_duplicates_keep_id(i, cleaned)
        if kept:
            cleaned_with_id.append(kept)

df_debug = pd.DataFrame(debug_log, columns=["doc_id", "status"])
print("\n📊 Filter summary:\n", df_debug["status"].value_counts())


# ============================================
# 5️⃣ Per-line invisible watermark analysis (1-to-1) + reduction tracking
# ============================================

def count_char(text: str, ch: str) -> int:
    """Count occurrences of a specific invisible char."""
    return text.count(ch)

per_doc_retention = []
for i, (orig_id, cleaned) in enumerate(cleaned_with_id):
    if i >= len(ZWC):
        break
    wm_char = ZWC[i]
    orig = texts[orig_id]
    orig_count = count_char(orig, wm_char)
    cleaned_count = count_char(cleaned, wm_char)
    reduced = cleaned_count < orig_count
    per_doc_retention.append({
        "doc_id": orig_id,
        "char_code": f"U+{ord(wm_char):04X}",
        "orig_count": orig_count,
        "cleaned_count": cleaned_count,
        "retention_ratio": (cleaned_count / orig_count) if orig_count > 0 else 0,
        "reduced": reduced,
    })

df_ret = pd.DataFrame(per_doc_retention).sort_values("doc_id").reset_index(drop=True)
retained = sum(df_ret["cleaned_count"] > 0)
retention_rate = retained / len(df_ret) * 100 if len(df_ret) else 0
reduced_docs = df_ret["reduced"].sum()

print(f"\n✅ 1-to-1 watermark detection complete.")
print(f"💧 Retained {retained}/{len(df_ret)} ({retention_rate:.2f}%) watermarks.")
print(f"📉 Watermark reduced in {reduced_docs} documents.")
print("\nSample result:")
print(df_ret.head(10))

plt.figure(figsize=(6, 4))
plt.hist(df_ret["retention_ratio"] * 100, bins=10, edgecolor="black")
plt.xlabel("Watermark retention (%)")
plt.ylabel("Document count")
plt.title("1-to-1 Invisible Watermark Retention per Document")
plt.grid(alpha=0.3)
plt.show()


In [1]:
!pip install tensorflow-datasets nltk langdetect pandas requests matplotlib tqdm
!pip install apache_beam
